# EDA: readmisión de pacientes con diabetes

Este notebook explora el dataset del Proyecto 2 para tomar las decisiones de limpieza
y confirmar los contratos (`src/schemas.py` y `src/config.py`).

Responde siete preguntas:

1. ¿Qué forma tiene el dataset y coincide con el contrato?
2. ¿Cómo se distribuye el target?
3. ¿Dónde hay nulos y qué columnas no sirven?
4. ¿Los valores de las categorías coinciden con los permitidos en `schemas.py`?
5. ¿Hay pacientes con varios encuentros?
6. ¿Qué encuentros no pueden tener readmisión?
7. ¿Cómo vienen los diagnósticos y cómo se agrupan?

El notebook solo explora y justifica. La limpieza definitiva vive en `src/preprocessing.py`,
que es lo que importa el DAG.

Después de cada sección hay una celda **Conclusión** para escribir, con nuestras palabras,
qué decidimos y por qué. Esas conclusiones van a `docs/decisiones.md`.

## 0. Preparación

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path
from typing import get_args

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

# El notebook vive en notebooks/; la raíz del repo es la carpeta de arriba.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import config
import schemas

# El CSV va en data/ (está en .gitignore, no se sube al repo).
DATA_FILE = Path(os.getenv("DATA_FILE", REPO_ROOT / "data" / "Diabetes" / "Diabetes.csv"))
print("Archivo de datos:", DATA_FILE)

In [ ]:
# Descarga el dataset del curso si todavía no está en la VM.
DRIVE_ID = "1k5-1caezQ3zWJbKaiMULTGq-3sz6uThC"
URL = f"https://drive.google.com/uc?export=download&id={DRIVE_ID}"

if not DATA_FILE.exists():
    DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
    print("Descargando...")
    urllib.request.urlretrieve(URL, DATA_FILE)

# Si Google Drive devolvió una página web en vez del CSV, avisamos.
with open(DATA_FILE, "rb") as f:
    first_bytes = f.read(200)
if first_bytes.lstrip().startswith(b"<"):
    DATA_FILE.unlink()
    raise RuntimeError(
        "Se descargó una página HTML, no el CSV. Descárgalo manualmente desde el enlace "
        f"del enunciado y déjalo en {DATA_FILE}"
    )
print(f"OK: {DATA_FILE.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Lectura controlada de nulos:
# - "?" es como el dataset original marca los datos faltantes.
# - keep_default_na=False evita que pandas convierta el texto "None" en nulo.
#   En A1Cresult y max_glu_serum, "None" significa "no se hizo la prueba", que es
#   información real, no un dato faltante.
df = pd.read_csv(DATA_FILE, na_values=["?", ""], keep_default_na=False, low_memory=False)
df.head()

## 1. Forma del dataset y comparación con el contrato

In [ ]:
print(f"Filas: {len(df):,}  |  Columnas: {df.shape[1]}")
print(f"Lotes de {config.BATCH_SIZE:,} registros necesarios: {-(-len(df) // config.BATCH_SIZE)}")
df.dtypes.value_counts()

In [ ]:
# ¿Están en el dataset las columnas que el contrato necesita?
needed = schemas.FEATURE_COLUMNS + [config.PATIENT_ID_COLUMN, config.TARGET_SOURCE_COLUMN]
missing = [c for c in needed if c not in df.columns]
extra = [c for c in df.columns if c not in needed]

print("Columnas del contrato que NO están en el dataset:", missing or "ninguna")
print(f"\nColumnas del dataset que el contrato no usa ({len(extra)}):")
print(extra)

In [ ]:
# ¿Hay encuentros duplicados?
if "encounter_id" in df.columns:
    print("encounter_id repetidos:", df["encounter_id"].duplicated().sum())
print("Filas completamente duplicadas:", df.duplicated().sum())

**Conclusión 1:** _(escribir después de correr)_

## 2. Target

In [ ]:
target_counts = df[config.TARGET_SOURCE_COLUMN].value_counts(dropna=False)
target_pct = (target_counts / len(df) * 100).round(2)
display(pd.DataFrame({"registros": target_counts, "%": target_pct}))

positive_rate = (df[config.TARGET_SOURCE_COLUMN] == config.POSITIVE_CLASS_VALUE).mean()
print(f"Clase positiva ({config.POSITIVE_CLASS_VALUE!r}): {positive_rate:.2%} de los registros")
print(f"Por cada readmisión <30 hay {(1 - positive_rate) / positive_rate:.1f} que no lo son")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
target_counts.plot.bar(ax=ax, color="#4C78A8")
ax.set_title("Distribución de readmitted")
ax.set_xlabel("")
ax.set_ylabel("Registros")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Con un desbalance así, *accuracy* engaña: un modelo que siempre dice "no se readmite"
tiene accuracy alta y no detecta a nadie. Por eso la métrica principal debe mirar la clase
positiva (recall, F2 o PR-AUC).

**Conclusión 2:** _(escribir: target binario sí/no, porcentaje de positivos y métrica elegida)_

## 3. Nulos, columnas constantes y columnas inútiles

In [ ]:
nulls = df.isna().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
null_table = pd.DataFrame({"nulos": nulls, "%": (nulls / len(df) * 100).round(2)})
null_table

In [ ]:
if len(null_table):
    fig, ax = plt.subplots(figsize=(7, 0.4 * len(null_table) + 1))
    null_table["%"].sort_values().plot.barh(ax=ax, color="#E45756")
    ax.axvline(40, color="gray", linestyle="--", linewidth=1)
    ax.set_title("% de nulos por columna (línea: 40%)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Columnas con un solo valor (no aportan nada) o casi constantes (>99% el mismo valor).
summary = []
for col in df.columns:
    top_share = df[col].value_counts(normalize=True, dropna=False).iloc[0]
    summary.append((col, df[col].nunique(dropna=False), round(top_share * 100, 2)))
const = pd.DataFrame(summary, columns=["columna", "valores_distintos", "% valor_más_común"])
const = const[const["% valor_más_común"] >= 99].sort_values("% valor_más_común", ascending=False)
const

In [ ]:
# Columnas candidatas a eliminar (regla: >40% nulos o >99% un solo valor).
# Los identificadores no son features: sirven para trazar y para el split.
high_null = null_table.index[null_table["%"] > 40].tolist()
near_constant = const["columna"].tolist()
identifiers = [c for c in ["encounter_id", "patient_nbr"] if c in df.columns]

print("Más de 40% nulos:", high_null)
print("Casi constantes:", near_constant)
print("Identificadores (no son features):", identifiers)

**Conclusión 3:** _(escribir: qué columnas salen y por qué; cómo se tratan los nulos que
quedan, por ejemplo race → "Unknown" y medical_specialty → "Unknown")_

## 4. Valores de las categorías frente al contrato

`schemas.py` define qué valores acepta la API en cada campo. Aquí comparamos esos valores
con los que realmente trae el dataset. `diag_*` y `medical_specialty` se omiten porque en el
contrato van **agrupados** (se revisan en la sección 7).

In [ ]:
grouped_in_contract = {"diag_1", "diag_2", "diag_3", "medical_specialty"}
rows = []
for name, field in schemas.PatientFeatures.model_fields.items():
    allowed = get_args(field.annotation)
    if not allowed or name in grouped_in_contract or name not in df.columns:
        continue
    real = set(df[name].dropna().astype(str).unique())
    allowed = {str(v) for v in allowed}
    rows.append({
        "campo": name,
        "en datos y NO en contrato": sorted(real - allowed),
        "en contrato y NO en datos": sorted(allowed - real),
        "nulos": int(df[name].isna().sum()),
    })
cat_check = pd.DataFrame(rows)
cat_check

In [ ]:
# Lo mismo para los campos numéricos: ¿los límites ge/le del contrato rechazarían datos reales?
rows = []
for name, field in schemas.PatientFeatures.model_fields.items():
    if name not in schemas.NUMERIC_FEATURES or name not in df.columns:
        continue
    lo = next((m.ge for m in field.metadata if hasattr(m, "ge")), None)
    hi = next((m.le for m in field.metadata if hasattr(m, "le")), None)
    col = pd.to_numeric(df[name], errors="coerce")
    rows.append({
        "campo": name, "mín real": col.min(), "máx real": col.max(),
        "contrato": f"[{lo}, {hi}]",
        "fuera de rango": int(((col < lo) | (col > hi)).sum()),
    })
pd.DataFrame(rows)

In [ ]:
# Distribución de cada categórica del contrato, para ver cuáles valores son raros.
for name in ["race", "gender", "age", "max_glu_serum", "A1Cresult", "insulin", "metformin", "change", "diabetesMed"]:
    if name in df.columns:
        print(f"--- {name}")
        print(df[name].value_counts(dropna=False).to_string(), "\n")

**Conclusión 4:** _(escribir: qué valores hay que agregar o quitar en `schemas.py`, y qué
hacer con valores raros, por ejemplo gender "Unknown/Invalid")_

## 5. Pacientes con varios encuentros

In [ ]:
pid = config.PATIENT_ID_COLUMN
per_patient = df.groupby(pid).size()
repeated = per_patient[per_patient > 1]

print(f"Encuentros: {len(df):,}")
print(f"Pacientes distintos: {per_patient.size:,}")
print(f"Pacientes con más de un encuentro: {repeated.size:,} ({repeated.size / per_patient.size:.1%})")
print(f"Encuentros que pertenecen a esos pacientes: {repeated.sum():,} ({repeated.sum() / len(df):.1%})")
print(f"Máximo de encuentros de un mismo paciente: {per_patient.max()}")

Si un paciente aparece en entrenamiento y en prueba, el modelo "ya lo conoce" y las métricas
salen infladas. Por eso el split se hace **por paciente**: todos los encuentros de un paciente
caen en el mismo conjunto. Además se hace con un hash del `patient_nbr` y no al azar, para
que en cada ejecución del DAG un paciente quede siempre en el mismo conjunto, aunque lleguen
lotes nuevos.

In [ ]:
# Prueba del split determinista por hash: proporciones obtenidas.
import hashlib

def split_of(patient_id) -> str:
    h = hashlib.md5(f"{config.SPLIT_SALT}-{patient_id}".encode()).hexdigest()
    u = int(h[:8], 16) / 0xFFFFFFFF  # número entre 0 y 1, siempre el mismo para ese paciente
    if u < config.TRAIN_FRACTION:
        return "train"
    if u < config.TRAIN_FRACTION + config.VAL_FRACTION:
        return "val"
    return "test"

split = df[pid].map(split_of)
positive = df[config.TARGET_SOURCE_COLUMN] == config.POSITIVE_CLASS_VALUE
pd.DataFrame({
    "encuentros": split.value_counts(),
    "% encuentros": (split.value_counts(normalize=True) * 100).round(1),
    "% positivos": (positive.groupby(split).mean() * 100).round(2),
})

**Conclusión 5:** _(escribir: el split por paciente queda balanceado en proporción y en % de positivos)_

## 6. Encuentros que no pueden tener readmisión

En el dataset original, `discharge_disposition_id` 11, 19, 20 y 21 significan que el paciente
falleció, y 13 y 14 que fue a cuidados paliativos (hospicio). Esos pacientes no pueden volver
a ingresar, así que dejarlos enseñaría al modelo algo falso.

In [ ]:
EXPIRED_OR_HOSPICE = [11, 13, 14, 19, 20, 21]

col = pd.to_numeric(df["discharge_disposition_id"], errors="coerce")
excluded = col.isin(EXPIRED_OR_HOSPICE)
print(f"Encuentros a excluir: {excluded.sum():,} ({excluded.mean():.2%})")

pd.DataFrame({
    "encuentros": col[excluded].value_counts(),
    "% readmitidos <30": (positive[excluded].groupby(col[excluded]).mean() * 100).round(2),
}).sort_index()

**Conclusión 6:** _(escribir: cuántos se excluyen y por qué)_

## 7. Diagnósticos (ICD-9) y especialidad médica

In [ ]:
# ¿Cómo vienen los códigos? Numéricos (428, 250.83), con V o E al inicio, o nulos.
def code_kind(value) -> str:
    if pd.isna(value):
        return "nulo"
    s = str(value).strip().upper()
    if s.startswith("V"):
        return "V (factores de salud)"
    if s.startswith("E"):
        return "E (causas externas)"
    try:
        float(s)
        return "numérico"
    except ValueError:
        return "otro"

pd.DataFrame({d: df[d].map(code_kind).value_counts() for d in ["diag_1", "diag_2", "diag_3"]}).fillna(0).astype(int)

Hay cientos de códigos distintos, demasiados para usarlos directo. Los agrupamos en las 9
categorías de Strack et al. (2014), el artículo que publicó este dataset, más `Missing` para
los diagnósticos que no vienen registrados.

La función **no se define aquí**: se importa de `src/preprocessing.py`, la misma que usa el
DAG. Así el EDA y el pipeline no pueden diferir.

In [ ]:
from preprocessing import group_icd9, group_specialty

diag_groups = pd.DataFrame({d: df[d].map(group_icd9).value_counts() for d in ["diag_1", "diag_2", "diag_3"]}).fillna(0).astype(int)
diag_groups

In [ ]:
# ¿El grupo del diagnóstico principal cambia la tasa de readmisión?
g = df["diag_1"].map(group_icd9)
(positive.groupby(g).mean() * 100).round(2).sort_values(ascending=False).rename("% readmitidos <30")

In [ ]:
# Especialidad médica: muchas categorías y muchos nulos.
spec = df["medical_specialty"].fillna("Unknown")
print(f"Especialidades distintas: {spec.nunique()}")
spec.value_counts().head(15)

In [ ]:
# Después de agrupar (src/preprocessing.py): 73 valores -> 7 grupos.
grouped_spec = df["medical_specialty"].map(group_specialty)
display(pd.DataFrame({
    "encuentros": grouped_spec.value_counts(),
    "% readmitidos <30": (positive.groupby(grouped_spec).mean() * 100).round(2),
}))

**Conclusión 7:** _(escribir: agrupación de diagnósticos, qué hacer con "Missing" —agregarlo al
contrato o mandarlo a "Other"— y cómo se agrupa medical_specialty)_

## 8. Variables que más se relacionan con el target

Una mirada rápida para el video: ¿qué variables parecen anticipar una readmisión?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

inp = pd.to_numeric(df["number_inpatient"], errors="coerce").clip(upper=5)
(positive.groupby(inp).mean() * 100).plot.bar(ax=axes[0], color="#4C78A8")
axes[0].set_title("% readmitidos <30 según hospitalizaciones previas")
axes[0].set_xlabel("number_inpatient (5 = 5 o más)")
axes[0].set_ylabel("%")

(positive.groupby(df["age"]).mean() * 100).plot.bar(ax=axes[1], color="#4C78A8")
axes[1].set_title("% readmitidos <30 por edad")
axes[1].set_xlabel("")

for ax in axes:
    ax.tick_params(axis="x", rotation=0 if ax is axes[0] else 45)
plt.tight_layout()
plt.show()

In [ ]:
# Correlación de las numéricas con el target binario.
numeric = df[schemas.NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce")
numeric.corrwith(positive.astype(int)).sort_values(ascending=False).round(3).rename("correlación con <30")

**Conclusión 8:** _(escribir: qué variables destacan)_

## 9. Resumen para `docs/decisiones.md`

Esta celda junta los números de arriba. Revísalos, ajusta lo que decidan y copia las líneas
a `docs/decisiones.md`.

In [ ]:
from datetime import date
MONTHS = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
today = f"{date.today().day}-{MONTHS[date.today().month - 1]}"

# Las features del contrato no se eliminan aunque tengan muchos nulos:
# sus nulos se convierten en una categoría ("Unknown").
candidates = set(high_null + near_constant)
drop_cols = sorted(candidates - set(schemas.FEATURE_COLUMNS))
kept_with_nulls = sorted(candidates & set(schemas.FEATURE_COLUMNS))
race_nulls = int(df["race"].isna().sum())
gender_invalid = int((~df["gender"].isin(["Male", "Female"])).sum())
n_specialties = df["medical_specialty"].nunique()
lines = [
    f"{today}. Dataset: {len(df):,} encuentros, {per_patient.size:,} pacientes, {df.shape[1]} columnas. "
    f"Se carga en {-(-len(df) // config.BATCH_SIZE)} lotes de máximo {config.BATCH_SIZE:,}.",
    f"{today}. Target binario: readmitted == '{config.POSITIVE_CLASS_VALUE}' (positivos: {positive_rate:.1%}). "
    "Accuracy no sirve por el desbalance.",
    f"{today}. Se eliminan columnas con >40% nulos o >99% un solo valor: {', '.join(drop_cols) or 'ninguna'}.",
    f"{today}. Se conservan aunque tengan muchos nulos (nulo -> 'Unknown'): {', '.join(kept_with_nulls) or 'ninguna'}.",
    f"{today}. El modelo usa solo las {len(schemas.FEATURE_COLUMNS)} features de src/schemas.py; quedan fuera los identificadores, "
    "weight, payer_code y los medicamentos salvo insulin y metformin.",
    f"{today}. encounter_id y patient_nbr no son features; patient_nbr se usa para el split.",
    f"{today}. race: los {race_nulls:,} nulos pasan a 'Unknown'. gender: se excluyen {gender_invalid} encuentros con 'Unknown/Invalid'.",
    f"{today}. A1Cresult y max_glu_serum: 'None' significa prueba no realizada y se conserva como categoría, no como nulo.",
    f"{today}. raw_data guarda todo tal como llega; las exclusiones se aplican solo al pasar a clean_data y antes del split.",
    f"{today}. Se excluyen {excluded.sum():,} encuentros de fallecidos u hospicio (discharge_disposition_id {EXPIRED_OR_HOSPICE}).",
    f"{today}. Split determinista por hash de patient_nbr: {config.TRAIN_FRACTION:.0%}/{config.VAL_FRACTION:.0%}/{config.TEST_FRACTION:.0%}; "
    f"{repeated.size:,} pacientes tienen varios encuentros y no pueden quedar en dos conjuntos.",
    f"{today}. Diagnósticos ICD-9 agrupados en 9 categorías (Strack et al., 2014) más 'Missing' para los no registrados.",
    f"{today}. medical_specialty ({n_specialties} valores) se agrupa en 7: cuatro especialidades frecuentes, Surgery, Other y Unknown.",
]
if missing:
    lines.append(f"{today}. ATENCIÓN: columnas del contrato que no existen en el dataset: {missing}.")
print("\n".join(lines))